In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import random
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

# --- 1. REPRODUCIBILITY & DEVICE ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- 2. HYPERPARAMETERS ---
LAMBDA = 10 
N_CRITIC = 5
LR = 1e-4
BETAS = (0.0, 0.9) 
SEQ_LEN = 128
EMBED_DIM = 128
HIDDEN_DIM = 256
BATCH_SIZE = 64
EPOCHS = 100

# --- 3. DATASET & VOCAB WITH 80/20 SPLIT ---
class URLDataset(Dataset):
    def __init__(self, urls, labels, char_to_int, vocab_size, seq_len=128):
        self.urls = urls
        self.labels = labels
        self.char_to_int = char_to_int
        self.vocab_size = vocab_size
        self.seq_len = seq_len

    def __len__(self):
        return len(self.urls)

    def __getitem__(self, idx):
        url = str(self.urls[idx])
        label = self.labels[idx]
        encoded = [self.char_to_int.get(char, 1) for char in url[:self.seq_len]]
        padding = [0] * (self.seq_len - len(encoded))
        encoded_tensor = torch.tensor(encoded + padding, dtype=torch.long)
        one_hot = torch.nn.functional.one_hot(encoded_tensor, num_classes=self.vocab_size).float()
        return one_hot, torch.tensor(label, dtype=torch.long)

# Data Loading and Splitting
print("Loading dataset...")
df = pd.read_csv('train_slim.csv')
df['label_idx'] = df['label'].apply(lambda x: 1 if str(x).lower() == 'phish' else 0)

# Build Global Vocab
all_chars = "".join(map(str, df['url'].values))
chars = sorted(list(set(all_chars)))
char_to_int = {ch: i + 2 for i, ch in enumerate(chars)}
char_to_int['<PAD>'] = 0
char_to_int['<UNK>'] = 1
vocab_size = len(char_to_int)
print(f"Vocabulary Size: {vocab_size}")

# 80/20 Split
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label_idx'])
print(f"Dataset Split: {len(train_df)} Training, {len(test_df)} Testing")

train_dataset = URLDataset(train_df['url'].values, train_df['label_idx'].values, char_to_int, vocab_size, SEQ_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# --- 4. MODELS ---
class Generator(nn.Module):
    def __init__(self, vocab_size, seq_len, embed_dim, hidden_dim):
        super().__init__()
        self.label_emb = nn.Embedding(2, embed_dim)
        self.gru = nn.GRU(100 + embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.seq_len = seq_len

    def forward(self, z, labels, temp=1.0):
        c = self.label_emb(labels).unsqueeze(1).repeat(1, self.seq_len, 1)
        z = z.unsqueeze(1).repeat(1, self.seq_len, 1)
        x = torch.cat([z, c], dim=2) 
        with torch.backends.cudnn.flags(enabled=False):
            out, _ = self.gru(x)
        logits = self.fc(out)
        return torch.nn.functional.gumbel_softmax(logits, tau=temp, hard=True)

class Discriminator(nn.Module):
    def __init__(self, vocab_size, seq_len, embed_dim):
        super().__init__()
        self.label_emb = nn.Embedding(2, embed_dim)
        self.conv = nn.Conv1d(vocab_size, 64, kernel_size=3, padding=1)
        self.ln = nn.LayerNorm([64, seq_len])
        self.lstm = nn.LSTM(64, 128, batch_first=True)
        self.fc = nn.Linear(128 + embed_dim, 1)

    def forward(self, x, labels):
        x = x.transpose(1, 2) 
        x = torch.relu(self.ln(self.conv(x))).transpose(1, 2)
        with torch.backends.cudnn.flags(enabled=False):
            _, (h, _) = self.lstm(x)
        c = self.label_emb(labels)
        combined = torch.cat([h[-1], c], dim=1)
        return self.fc(combined)

# --- 5. TRAINING UTILS ---
def compute_gradient_penalty(D, real_samples, fake_samples, labels):
    alpha = torch.rand(real_samples.size(0), 1, 1).to(device)
    interpolates = (alpha * real_samples + (1 - alpha) * fake_samples).requires_grad_(True)
    d_interpolates = D(interpolates, labels)
    fake = torch.ones(d_interpolates.size()).to(device)
    gradients = torch.autograd.grad(
        outputs=d_interpolates, inputs=interpolates,
        grad_outputs=fake, create_graph=True, retain_graph=True, only_inputs=True
    )[0]
    gradients = gradients.reshape(gradients.size(0), -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

def generate_readable_urls(generator, char_to_int_map, label_type, num_samples=2):
    generator.eval()
    int_to_char = {i: ch for ch, i in char_to_int_map.items()}
    with torch.no_grad():
        z = torch.randn(num_samples, 100).to(device)
        labels = torch.full((num_samples,), label_type, dtype=torch.long).to(device)
        fake_one_hot = generator(z, labels, temp=0.5) 
        char_indices = torch.argmax(fake_one_hot, dim=2).cpu().numpy()
        urls = ["".join([int_to_char[i] for i in seq if i > 1]) for seq in char_indices]
    generator.train()
    return urls

# --- 6. THE VERBOSE TRAINING LOOP ---
G = Generator(vocab_size, SEQ_LEN, EMBED_DIM, HIDDEN_DIM).to(device)
D = Discriminator(vocab_size, SEQ_LEN, EMBED_DIM).to(device)

opt_G = optim.Adam(G.parameters(), lr=LR, betas=BETAS)
opt_D = optim.Adam(D.parameters(), lr=LR, betas=BETAS)

print("\n--- Starting Training Process ---")
for epoch in range(EPOCHS):
    d_losses = []
    g_losses = []
    
    for i, (real_urls, labels) in enumerate(train_loader):
        real_urls, labels = real_urls.to(device), labels.to(device)
        batch_size = real_urls.size(0)

        # --- TRAIN DISCRIMINATOR ---
        opt_D.zero_grad()
        z = torch.randn(batch_size, 100).to(device)
        fake_urls = G(z, labels)
        
        real_validity = D(real_urls, labels)
        fake_validity = D(fake_urls.detach(), labels)
        
        gp = compute_gradient_penalty(D, real_urls, fake_urls.detach(), labels)
        d_loss = -torch.mean(real_validity) + torch.mean(fake_validity) + LAMBDA * gp
        
        d_loss.backward()
        opt_D.step()
        d_losses.append(d_loss.item())

        # --- TRAIN GENERATOR ---
        if i % N_CRITIC == 0:
            opt_G.zero_grad()
            gen_urls = G(z, labels)
            g_loss = -torch.mean(D(gen_urls, labels))
            g_loss.backward()
            opt_G.step()
            g_losses.append(g_loss.item())

        if i % 50 == 0:
            print(f"[Epoch {epoch}/{EPOCHS}] [Batch {i}/{len(train_loader)}] D_loss: {d_loss.item():.4f}", end='\r')

    # End of Epoch Verbosity
    avg_d = np.mean(d_losses)
    avg_g = np.mean(g_losses) if g_losses else 0
    print(f"\n>> Epoch {epoch} Summary: Avg D_Loss: {avg_d:.4f} | Avg G_Loss: {avg_g:.4f}")
    
    if epoch % 10 == 0:
        print("\n--- Generating Periodic Samples ---")
        print(f"  Benign: {generate_readable_urls(G, char_to_int, 0, 1)}")
        print(f"  Phishing: {generate_readable_urls(G, char_to_int, 1, 1)}")
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': G.state_dict(),
            'char_to_int': char_to_int,
            'vocab_size': vocab_size,
            'avg_d_loss': avg_d
        }
        torch.save(checkpoint, f"url_gen_verbose_epoch_{epoch}.pt")
        print(f"--- Checkpoint saved for epoch {epoch} ---\n")

print("\nTraining Complete.")

Using device: cuda
Loading dataset...
Vocabulary Size: 162
Dataset Split: 90851 Training, 22713 Testing

--- Starting Training Process ---


c:\Users\yadhu\Documents\Deeplearning_Backyard\Aegis_v1\phish\Lib\site-packages\torch\autograd\graph.py:823: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\cuda\CublasHandlePool.cpp:180.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[Epoch 0/100] [Batch 1400/1420] D_loss: -2.9082
>> Epoch 0 Summary: Avg D_Loss: -2.6673 | Avg G_Loss: -0.3009

--- Generating Periodic Samples ---
  Benign: ['\x81r¬Zk\x8a¶y13s1IJS8H𝕢pXn:3pn8y-xyxynoz𝕝nff!s1nmp/ppGdhp#§m.bcwml06fì7p\x9bpu¼Âb']
  Phishing: ['°7or¥𝕫MiQFI𝕨1𝕥1i,90i\x80t¬\x8b4w¬∕ntr³4ê#v&fAkx6mp__9\x94/nyek[j¬lN']
--- Checkpoint saved for epoch 0 ---

[Epoch 1/100] [Batch 1400/1420] D_loss: -0.6566
>> Epoch 1 Summary: Avg D_Loss: -1.5157 | Avg G_Loss: -1.3119
[Epoch 2/100] [Batch 1400/1420] D_loss: -3.0362
>> Epoch 2 Summary: Avg D_Loss: -2.6492 | Avg G_Loss: -1.7673
[Epoch 3/100] [Batch 1400/1420] D_loss: -5.1012
>> Epoch 3 Summary: Avg D_Loss: -3.9842 | Avg G_Loss: -1.3005
[Epoch 4/100] [Batch 1400/1420] D_loss: -5.6603
>> Epoch 4 Summary: Avg D_Loss: -4.7773 | Avg G_Loss: -1.1014
[Epoch 5/100] [Batch 1400/1420] D_loss: -5.9162
>> Epoch 5 Summary: Avg D_Loss: -5.4580 | Avg G_Loss: -0.8951
[Epoch 6/100] [Batch 1400/1420] D_loss: -6.7194
>> Epoch 6 Summary: Avg D_Loss: -5.7